In [1]:
import pandas as pd

data_fname = "../outputdata/clips_metadata_with_patterns.csv"
df = pd.read_csv(data_fname)
df


/tmp/ipykernel_89584/1453928005.py:4: DtypeWarning: Columns (1,6,8,26,31,32,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_fname)


,Model,StateFileName,LevelFullName,Scene,ClipCode,TotalFrames,Bk2Filepath,GameName,Subject,World,...,Pattern_Empty stair valley,Pattern_Enemy stair valley,Pattern_Gap stair valley,Pattern_Reward,Pattern_Moving platform,Pattern_Flagpole,Pattern_Beginning,Pattern_Bonus zone,Pattern_Waterworld,Learning_Phase
0,human,NaN,w1l3,0,103100000111,2628,NaN,SuperMarioBros-Nes,sub-01,1,...,0,0,0,0,0,0,1,0,0,Late discovery
1,human,NaN,w1l1,10,101000003476,4831,NaN,SuperMarioBros-Nes,sub-01,1,...,1,0,0,0,0,0,0,0,0,Early discovery
2,human,NaN,w1l3,6,103010001275,2528,NaN,SuperMarioBros-Nes,sub-01,1,...,0,0,0,0,1,0,0,0,0,Early discovery
3,human,NaN,w2l1,13,104000002344,3354,NaN,SuperMarioBros-Nes,sub-01,2,...,0,0,0,0,0,0,0,0,0,Early discovery
4,human,NaN,w1l2,6,102010002114,3090,NaN,SuperMarioBros-Nes,sub-01,1,...,0,0,0,0,0,0,0,0,0,Early discovery
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336841,ep-8000,scene_clips/sub-06/ses-025/beh/savestates/sub-...,w8l2,4,2505020000896,206,ppo_mario_ep-8000/sub-06/ses-025/beh/bk2/sub-0...,SuperMarioBros-Nes,6,8,...,0,0,0,1,0,0,0,0,0,NaN
336842,ep-8000,scene_clips/sub-06/ses-025/beh/savestates/sub-...,w1l3,0,2504040000111,103,ppo_mario_ep-8000/sub-06/ses-025/beh/bk2/sub-0...,SuperMarioBros-Nes,6,1,...,0,0,0,0,0,0,1,0,0,NaN
336843,ep-8000,scene_clips/sub-06/ses-025/beh/savestates/sub-...,w6l3,8,2501030004606,339,ppo_mario_ep-8000/sub-06/ses-025/beh/bk2/sub-0...,SuperMarioBros-Nes,6,6,...,0,0,0,0,1,0,0,0,0,NaN
336844,ep-8000,scene_clips/sub-06/ses-025/beh/savestates/sub-...,w7l1,7,2503010002929,428,ppo_mario_ep-8000/sub-06/ses-025/beh/bk2/sub-0...,SuperMarioBros-Nes,6,7,...,0,0,0,0,0,0,0,1,0,NaN


In [2]:
level = 'w1l2'
scene = 1
subject = 'sub-01'
scene_fullname = f"{level}s{scene}"

df_filtered = df[(df['LevelFullName'] == level) & (df['Scene'] == scene) & (df['Subject'] == subject)]




In [7]:
# %pip install plotly ipywidgets pillow

import os
import numpy as np
import pandas as pd
from PIL import Image

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale

import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- DATA ----------
# Keep your existing df; otherwise you can load:
# df = pd.read_csv("/mnt/data/tmp_filtered_clips.csv")

df_human = df[df["Model"] == "human"].copy()

metrics = [
    "Duration", "Cleared", "ScoreGained", "X_Traveled", "Lives_lost",
    "Hits_taken", "Enemies_killed", "Powerups_collected", "Bricks_destroyed",
    "CoinsGained", "Average_speed"
]

phase_order = ["Early discovery", "Late discovery", "Early practice", "Late practice"]
phase_colors = dict(zip(phase_order, sample_colorscale("Viridis", [0.00, 0.33, 0.66, 1.00])))

# Z-score across the entire human dataset (per metric)
means = df_human[metrics].mean(numeric_only=True)
stds  = df_human[metrics].std(numeric_only=True).replace(0, np.nan)
z_cols = []
for m in metrics:
    zc = f"{m}_z"
    z_cols.append(zc)
    df_human[zc] = (df_human[m] - means[m]) / stds[m]

# ---------- HELPERS ----------
def scenes_for_level(level):
    return sorted(df_human.loc[df_human["LevelFullName"] == level, "Scene"].dropna().unique().tolist())

def subjects_for(level, scene):
    m = (df_human["LevelFullName"] == level) & (df_human["Scene"] == scene)
    return sorted(df_human.loc[m, "Subject"].dropna().unique().tolist())

def get_subset(level, scene, subject):
    m = (
        (df_human["LevelFullName"] == level) &
        (df_human["Scene"] == scene) &
        (df_human["Subject"] == subject)
    )
    return df_human.loc[m].copy()

def load_scene_image(level, scene):
    scene_fullname = f"{level}s{scene}"
    path = f"../../mario.scenes/sourcedata/scene_backgrounds/{scene_fullname}.png"
    if os.path.exists(path):
        im = Image.open(path).convert("RGBA")
        return np.array(im), path
    return None, path

def make_dashboard(level, scene, subject):
    sub = get_subset(level, scene, subject)

    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{"type": "xy"}, {"type": "polar"}, {"type": "xy"}]],
        column_widths=[0.34, 0.33, 0.33],
        horizontal_spacing=0.06,
        subplot_titles=("Scene Background", "Learning-Phase Averages (Z-scored)", "Clips per Learning Phase")
    )

    # Left: non-interactive background
    img_arr, img_path = load_scene_image(level, scene)
    if img_arr is not None:
        fig.add_trace(go.Image(z=img_arr, hoverinfo="skip"), row=1, col=1)
    else:
        fig.add_annotation(row=1, col=1, x=0.5, y=0.5, xref="x1", yref="y1",
                           text=f"No image found:<br>{img_path}", showarrow=False)
    fig.update_xaxes(visible=False, fixedrange=True, row=1, col=1)
    fig.update_yaxes(visible=False, fixedrange=True, row=1, col=1)

    # Middle: polar (z), hover shows raw mean too
    if not sub.empty:
        raw_means = sub.groupby("Learning_Phase")[metrics].mean(numeric_only=True).reindex(phase_order)
        z_means   = sub.groupby("Learning_Phase")[z_cols].mean(numeric_only=True).reindex(phase_order)

        for phase in phase_order:
            if phase in z_means.index:
                z_vals   = z_means.loc[phase].reindex([f"{m}_z" for m in metrics]).values.astype(float).tolist()
                raw_vals = raw_means.loc[phase].reindex(metrics).values.astype(float).tolist()
                if np.all(np.isnan(z_vals)):
                    continue

                z_closed   = z_vals + [z_vals[0]]
                raw_closed = raw_vals + [raw_vals[0]]
                theta      = metrics + [metrics[0]]
                customdata = np.array(raw_closed).reshape(-1, 1)

                fig.add_trace(
                    go.Scatterpolar(
                        r=z_closed,
                        theta=theta,
                        mode="lines+markers",
                        name=phase,
                        line=dict(color=phase_colors[phase]),
                        customdata=customdata,
                        hovertemplate="%{theta}<br>z: %{r:.2f}<br>raw mean: %{customdata[0]:.3g}<extra>"+phase+"</extra>",
                    ),
                    row=1, col=2
                )

        fig.update_polars(radialaxis=dict(showgrid=True), angularaxis=dict(direction="clockwise"))

    # Right: bar (counts) with outside labels
    counts = sub["Learning_Phase"].value_counts().reindex(phase_order).fillna(0).astype(int)
    x_vals = counts.index.tolist()
    y_vals = counts.values.tolist()
    fig.add_trace(
        go.Bar(
            x=x_vals, y=y_vals,
            marker_color=[phase_colors[p] for p in x_vals],
            text=y_vals, textposition="outside",
            cliponaxis=False,
            hovertemplate="Phase: %{x}<br>N clips: %{y}<extra></extra>",
        ),
        row=1, col=3
    )
    y_max = max(y_vals) if len(y_vals) else 0
    fig.update_yaxes(title_text="N clips", range=[0, y_max * 1.25 + (1 if y_max < 5 else 0)], row=1, col=3)
    fig.update_xaxes(title_text="Learning Phase", row=1, col=3)

    # Global layout tuned for clean initial render
    scene_fullname = f"{level}s{scene}"
    fig.update_layout(
        height=540,
        autosize=True,
        margin=dict(l=20, r=20, t=70, b=70),
        title_text=f"Level: {level} | Scene: {scene} ({scene_fullname}) | Subject: {subject}",
        legend=dict(orientation="h", y=-0.22, x=0.0),
        uniformtext_minsize=10, uniformtext_mode='hide',
        bargap=0.3,
    )
    # Shrink subplot titles, nudge slightly
    for ann in fig.layout.annotations:
        ann.font.size = 12
        ann.yshift = 6

    return fig

# ---------- WIDGETS ----------
levels = sorted(df_human["LevelFullName"].dropna().unique())

level_dd = widgets.Dropdown(
    options=levels, value=levels[0],
    description="Level:", layout=widgets.Layout(width="240px")
)

# Scene as SelectionSlider with exactly N positions (available scenes for the level)
def build_scene_slider(level):
    opts = scenes_for_level(level)
    if not opts:
        return widgets.SelectionSlider(options=[], value=None, description="Scene:", layout=widgets.Layout(width="500px"))
    # label each tick with the scene number; N positions = len(opts)
    return widgets.SelectionSlider(
        options=[(str(s), s) for s in opts],
        value=opts[0],
        description="Scene:",
        layout=widgets.Layout(width="500px"),
        continuous_update=False
    )

scene_slider = build_scene_slider(level_dd.value)

def current_subject_options():
    if scene_slider.value is None:
        return []
    return subjects_for(level_dd.value, scene_slider.value)

subject_dd = widgets.Dropdown(
    options=current_subject_options(),
    value=current_subject_options()[0] if current_subject_options() else None,
    description="Subject:", layout=widgets.Layout(width="240px")
)

out = widgets.Output(layout=widgets.Layout(border="0px"))

# ---------- CALLBACKS ----------
def on_level_change(change):
    # rebuild scene slider with N positions for the selected level
    global scene_slider
    new_slider = build_scene_slider(level_dd.value)
    # swap the widget in place inside the UI
    controls.children = (level_dd, new_slider, subject_dd)
    # rebind handler
    new_slider.observe(on_scene_change, names="value")
    scene_slider.unobserve(on_scene_change, names="value")
    # update reference
    scene_slider = new_slider

    # update subject list
    subj_opts = current_subject_options()
    subject_dd.options = subj_opts
    subject_dd.value = subj_opts[0] if subj_opts else None

    draw()

def on_scene_change(change):
    subj_opts = current_subject_options()
    subject_dd.options = subj_opts
    subject_dd.value = subj_opts[0] if subj_opts else None
    draw()

def on_subject_change(change):
    draw()

def draw():
    with out:
        clear_output(wait=True)
        if (level_dd.value is None) or (scene_slider.value is None) or (subject_dd.value is None):
            print("No data available for the current selection.")
            return
        fig = make_dashboard(level_dd.value, scene_slider.value, subject_dd.value)
        # Use .show with minimal toolbar & no selection tools
        fig.show(config={"displaylogo": False, "modeBarButtonsToRemove": ["select", "lasso2d"]})

# Wire up observers
level_dd.observe(on_level_change, names="value")
scene_slider.observe(on_scene_change, names="value")
subject_dd.observe(on_subject_change, names="value")

# ---------- DISPLAY FIRST, THEN DRAW (prevents initial overlap) ----------
controls = widgets.HBox([level_dd, scene_slider, subject_dd])
display(controls, out)
draw()  # clean initial render


Output(layout=Layout(border_bottom='0px', border_left='0px', border_right='0px', border_top='0px'))